# External workflow logs — tradespace sweep

Tutorial notebook for **Scenario B**: log many external runs under one **campaign title**, sweep design parameters with **pytest**, and use `status` to find the winning candidate.

You will:

1. Get or create a system with a tracked **L-bracket** STEP model (same `SYSTEM_NAME` as [Scenario A](workflow_log_scenario_a.ipynb))
2. Sweep fillet radius and wall thickness across a parameter grid
3. Upload parameters and JUnit results for each attempt
4. List campaign entries filtered by title
5. Build a new STEP revision from the winning log parameters and commit it for Open CAD extract

Companion to [Scenario A — design loop](workflow_log_scenario_a.ipynb).

> **External workflows vs jobs.** External workflows run on your machine or in CI; Istari Digital stores the outputs and log entry. [Istari jobs](https://docs.istaridigital.com/developers/SDK/api_reference/03-jobs) run on platform agents and are scheduled end-to-end by the platform.

Highlights:

- Multiple entries may share the same title — each is an independent record
- `status` plus attached parameter files identify the design point that passed

### Prerequisites

From the cookbook root, copy the first command cell into a terminal:

| Group | Packages | Used for |
|---|---|---|
| **`dev`** | `istari-digital-client`, `python-dotenv` | Connect, systems, workflow log API |
| **`advanced`** | `cadquery`, `pandas`, `jinja2`, `matplotlib`, `numpy`, `pytest`, `ipython` | Real STEP solids, tables, plots, tradespace sweep |

Other recipes need only `uv sync --group dev`.

- **Registry Service > 10.17.3** (2026-05 release or later)
- Credentials in [`samples/.env`](../.env): `ISTARI_REGISTRY_URL`, `ISTARI_PERSONAL_ACCESS_TOKEN`
- **Experimental features** (Istari Digital web app → **Application Settings** → **Experimental Features**):
  - **Branching** — required for `commit_changes()` (snapshot + baseline tag) in §2
  - **Workflow Log** — required for workflow outputs and log entries
- **`istari-digital-client` must be API-compatible with the registry** — Connect runs `Client.check_compatibility()` (SDK and registry version strings differ)

### Install kernel (optional)

After sync, copy the second command cell into a terminal. Select **Python (istari-client-cookbook)** in the kernel picker.

### Running order

Run cells top to bottom. Pause at the callout in §5 to review entries in the **Workflow log** tab.


In [ ]:
uv sync --group dev --group advanced


In [ ]:
uv run python -m ipykernel install --user --name istari-client-cookbook --display-name "Python (istari-client-cookbook)"


## Scenario

The part is a **mounting L-bracket** (base + upright flange with a circular boss). A fillet/thickness grid is evaluated with **pytest** outside the platform. Each attempt that runs uploads its parameter set and JUnit report under one shared campaign title. The first passing design stops the sweep (so only part of the grid is logged). Then the notebook rebuilds a real STEP solid from the winning parameters so you can run **Open CAD** `@istari:extract` (FreeCAD) and view the 3D geometry.

Each attempt becomes a **workflow log entry** linked to the active **configuration**.


## 1. Connect

Load credentials from [`samples/.env`](../.env). Two clients share one `Configuration`:

- `Client` — read the system, download tracked files, update models
- `V3Client` — workflow outputs and workflow log entries

Connect checks registry release level and SDK API compatibility via `check_compatibility()`.

Requires Registry Service **> 10.17.3** (2026-05+). Connect reads `X-Istari-Registry-Version` and fails fast otherwise.


In [ ]:
import os
import json
import re
from importlib.metadata import version as pkg_version
from pathlib import Path
from dotenv import load_dotenv

from istari_helpers import MINIMUM_REGISTRY_VERSION, registry_meets_minimum

from istari_digital_client import Client, V3Client, Configuration
from istari_digital_client.v3.models import WorkflowLogEntryCreateDto

load_dotenv("../.env")
REGISTRY_URL = os.environ["ISTARI_REGISTRY_URL"]
PAT = os.environ["ISTARI_PERSONAL_ACCESS_TOKEN"]

_match = re.match(r"^(https?://)(?:fileservice-v2\.)?(.+?)/?$", REGISTRY_URL)
UI_URL = REGISTRY_URL.rstrip("/") if not _match else f"{_match.group(1)}{_match.group(2)}"

config = Configuration(registry_url=REGISTRY_URL, registry_auth_token=PAT)
client = Client(config)     # read system + download files
v3     = V3Client(config)   # workflow outputs + workflow log

installed = pkg_version("istari-digital-client")

compat = client.check_compatibility()
assert compat and compat.server_version, (
    "Registry did not return compatibility headers. "
    "Check ISTARI_REGISTRY_URL and ISTARI_PERSONAL_ACCESS_TOKEN in samples/.env."
)
registry_version = compat.server_version

assert registry_meets_minimum(registry_version), (
    f"Istari Registry v{registry_version} does not meet the minimum for this recipe "
    f"(> {MINIMUM_REGISTRY_VERSION}, 2026-05 release). Point ISTARI_REGISTRY_URL at a "
    "2026-05+ deployment and install a matching istari-digital-client."
)

print("Connected to:", REGISTRY_URL)
print("UI:", UI_URL)
print(f"istari-digital-client {installed}")
print(f"Istari Registry v{registry_version} — compatible")
# print user email using v2 client


## 2. Get or create the demo system

Set `SYSTEM_NAME`, look up an active system with that name, and create it (with a `baseline` configuration) only when missing. Reuses the system from [Scenario A](workflow_log_scenario_a.ipynb) when you have already run it. Ends with `commit_changes()` so the baseline branch shows the tracked files.


In [ ]:
import bracket_step
import istari_helpers
from istari_digital_client.v2.models.new_system import NewSystem
from istari_digital_client.v2.models.new_system_configuration import NewSystemConfiguration
from istari_digital_client.v2.models.new_tracked_file import NewTrackedFile
from istari_digital_client.v2.models.tracked_file_specifier_type import TrackedFileSpecifierType

SYSTEM_NAME = "Workflow Log Walkthrough"
CONFIG_NAME = "baseline"

work = Path("_notebook_run")
work.mkdir(exist_ok=True)
src = work / "bracket.step"
# Real Open CASCADE L-bracket solid (R3 fillet / 3 mm wall) for FreeCAD extract.
bracket_step.write_bracket(src, fillet_radius_mm=3.0, wall_thickness_mm=3.0)

system = istari_helpers.find_system_by_name(client, SYSTEM_NAME)
if system is None:
    model = client.add_model(path=src,
                             description="Bracket — initial R3 design")
    system = client.create_system(NewSystem(
        name=SYSTEM_NAME,
        description="External workflow log demo",
    ))
    config_obj = client.create_configuration(
        system_id=system.id,
        new_system_configuration=NewSystemConfiguration(
            name=CONFIG_NAME,
            tracked_files=[NewTrackedFile(
                specifier_type=TrackedFileSpecifierType.LATEST,
                file_id=model.file.id,
            )],
        ),
    )
    MODEL_ID, FILE_ID = model.id, model.file.id
    print("Created system:", system.id)
else:
    config_obj = istari_helpers.find_configuration(client, system.id, CONFIG_NAME)
    if config_obj is None:
        raise RuntimeError(f"System {SYSTEM_NAME!r} exists but has no {CONFIG_NAME!r} configuration")
    tracked = client.list_tracked_files(configuration_id=config_obj.id).items
    if not tracked:
        raise RuntimeError(f"Configuration {CONFIG_NAME!r} has no tracked files")
    FILE_ID = tracked[0].file_id
    MODEL_ID = istari_helpers.model_id_for_file(client, FILE_ID)
    if MODEL_ID is None:
        raise RuntimeError(f"No model found for file {FILE_ID}")
    print("Using existing system:", system.id)

SYSTEM_ID, CONFIG_ID = system.id, config_obj.id
istari_helpers.commit_changes(client, SYSTEM_ID, CONFIG_ID)
print("System:", f"{UI_URL}/systems/{SYSTEM_ID}")

## 3. Requirement suite

`tradespace_tests.py` encodes stress, mass, and deflection limits. Parameters arrive via environment variables; `record_property` embeds them in the JUnit report.


In [ ]:
print(Path("tradespace_tests.py").read_text())

## 4. Run the sweep

For each candidate: write `parameters.json`, run pytest, upload outputs, log an entry under the campaign title. Stop at the first `SUCCESS`.


In [ ]:
import subprocess
import sys
import math
import pandas as pd

campaign = "Tradespace sweep — bracket fillet/thickness study"
grid = [(2, 3), (2, 5), (3, 3), (8, 2), (3, 6), (4, 6), (10, 2), (4, 3), (5, 3), (5, 4)]
ts_dir = work / "tradespace"
ts_dir.mkdir(exist_ok=True)


def evaluate(r, t):
    peak = 1500.0 / (math.sqrt(r) * t)
    return {"peak_stress_MPa": round(peak, 1), "safety_factor": round(276.0 / peak, 2),
            "mass_kg": round(0.30 + 0.10 * t + 0.03 * r, 2), "stiffness_index": round(r * t, 1)}


sweep_rows, winner = [], None
for attempt, (r, t) in enumerate(grid, 1):
    run_dir = ts_dir / f"cand_{attempt:02d}"
    run_dir.mkdir(exist_ok=True)

    # 1. the candidate parameter set — recorded as a first-class output on the entry
    metrics = evaluate(r, t)
    params = {"attempt": attempt, "campaign": campaign,
              "parameters": {"fillet_radius_mm": r, "wall_thickness_mm": t}, "metrics": metrics}
    params_path = run_dir / "00_parameters.json"
    params_path.write_text(json.dumps(params, indent=2))

    # 2. run the pytest requirement suite for this candidate -> real JUnit report
    junit = run_dir / "pytest_results.xml"
    proc = subprocess.run(
        [sys.executable, "-m", "pytest", "tradespace_tests.py", "-q", f"--junitxml={junit}"],
        env={**os.environ, "TS_FILLET_MM": str(r), "TS_THICKNESS_MM": str(t)},
        capture_output=True, text=True)
    passed = proc.returncode == 0
    status = "SUCCESS" if passed else "FAILED"

    # 3. upload outputs; the parameters file carries the design point on the entry
    summary = f"fillet={r} mm, thickness={t} mm"
    param_id = v3.create_workflow_output(system_id=SYSTEM_ID, path=params_path,
                   display_name=f"parameters ({summary})",
                   description=f"Candidate {attempt}: {summary}").id
    junit_id = v3.create_workflow_output(system_id=SYSTEM_ID, path=junit).id

    # 4. log this attempt under the SAME campaign title
    v3.create_workflow_log_entry(system_id=SYSTEM_ID,
        workflow_log_entry_create_dto=WorkflowLogEntryCreateDto(
            title=campaign, status=status, configuration_id=CONFIG_ID,
            workflow_output_ids=[param_id, junit_id]))

    sweep_rows.append({"attempt": attempt, "fillet_mm": r, "thickness_mm": t,
                       "SF": metrics["safety_factor"], "mass_kg": metrics["mass_kg"],
                       "stiffness": metrics["stiffness_index"], "result": status})
    print(f"  attempt {attempt:2d}: fillet={r:>2} thickness={t} -> {status}")
    if passed:
        winner = params["parameters"]
        break

print("\nWinning configuration:", winner)

Sweep summary — metrics and verdicts for each candidate:

In [ ]:
def color_status(v):
    return ("color:#15803d;font-weight:600" if v == "SUCCESS"
            else "color:#dc2626;font-weight:600" if v == "FAILED" else "")


pd.DataFrame(sweep_rows).style.map(color_status, subset=["result"])

## 5. Review the campaign

Filter entries by campaign title. The `SUCCESS` entry's parameters output records the winning fillet and thickness.

> **Pause here.** In the web app, open the campaign's `SUCCESS` entry and download the parameters file.


In [ ]:
page = v3.list_workflow_log_entries(system_id=SYSTEM_ID, title=[campaign], size=100)
passed_n = sum(e.status == "SUCCESS" for e in page.items)
print(f"{len(page.items)} entries share the campaign title — {passed_n} SUCCESS, {len(page.items) - passed_n} FAILED")

pd.DataFrame([{"status": e.status, "files": e.file_count,
               "created": e.created.strftime("%H:%M:%S"), "entry": str(e.id)[:8]}
              for e in page.items]).style.map(color_status, subset=["status"])

## 6. Apply the winning design from the workflow log

Take the `SUCCESS` parameter set, generate a real L-bracket STEP at those dimensions, and upload it as a **new model revision**. `commit_changes()` snapshots the configuration so the baseline branch shows the winning geometry.

> **Pause here.** Open the model in the Istari Digital web app → run **`@istari:extract`** with tool **`freecad`** (Open CAD). When the job completes, open `geometry_obj` / views to inspect the thicker/thinner plates and mounting boss sized by the winning fillet.

In [ ]:
# `winner` is set by the sweep cell (first SUCCESS). Re-run §4 if this fails.
assert winner is not None, "No SUCCESS candidate — re-run the sweep cell in §4."

fillet_mm = float(winner["fillet_radius_mm"])
thickness_mm = float(winner["wall_thickness_mm"])
print(f"Winning design from workflow log: fillet={fillet_mm} mm, thickness={thickness_mm} mm")

bracket_step.write_bracket(src, fillet_radius_mm=fillet_mm, wall_thickness_mm=thickness_mm)
client.update_model(
    model_id=MODEL_ID,
    path=src,
    description=f"Bracket — winning design R{fillet_mm:g} / t{thickness_mm:g} mm",
)
istari_helpers.commit_changes(client, SYSTEM_ID, CONFIG_ID)

print(f"Uploaded new revision of bracket.step")
print(f"Model: {UI_URL}/resources/{MODEL_ID}")
print(f"System: {UI_URL}/systems/{SYSTEM_ID}")
print(
    "Next: Artifacts → @istari:extract → tool freecad "
    "(see https://docs.istaridigital.com/integrations/CAD/open_cad)."
)

## 7. Recap

This notebook swept a grid of L-bracket fillet/thickness candidates with a local pytest suite. Each candidate that ran became a **workflow log entry** under one shared **campaign title**, with its parameter file, JUnit report, and `FAILED` / `SUCCESS` status. The loop stops at the first pass, so you typically see a handful of entries (not every row in the grid). Section 6 turns that `SUCCESS` parameter set into a new STEP revision you can extract and view in 3D.

**How to use the log in the Istari Digital web app**

1. Open the system → **Workflow log** tab.
2. Filter or scan by the campaign title to see only this study.
3. Open a `FAILED` entry to see which checks failed and which parameters were tried.
4. Open the `SUCCESS` entry and download the attached parameters file — that is the design point that met the requirements.
5. Open the model (`bracket.step`) → run **Open CAD** `@istari:extract` (`freecad`) on the revision uploaded in §6.

**Why log this on Istari Digital**

The sweep ran on your machine; Istari Digital keeps the durable record: every attempt tied to the **configuration** that was active, with outputs and pass/fail status queryable later. You can reopen the campaign, recover the winning parameters from the log, regenerate or re-upload geometry, and extract a 3D view — without hunting through local folders or CI history.


Open the **Workflow log** tab:

In [ ]:
print("Review both scenarios in the Workflow log tab:")
print(f"  {UI_URL}/systems/{SYSTEM_ID}")

### Learn more

- [External workflow logs](https://docs.istaridigital.com/developers/SDK/v3/03-workflow-logs) — SDK reference for `create_workflow_output`, `create_workflow_log_entry`, and listing entries
- [Open CAD Module](https://docs.istaridigital.com/integrations/CAD/open_cad) — `@istari:extract` with FreeCAD on STEP/IGES
- **Workflow log** tab on a system (requires the experimental feature above)


## Teardown (optional)

Archive the demo system when you are finished so repeated runs do not clutter the instance. Safe to skip during a live walkthrough; archiving is reversible.


In [ ]:
client.archive_system(system_id=SYSTEM_ID)
print("Archived system", SYSTEM_ID)